<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Create a Local Ethernet (Layer 2) Network: User-Defined Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to create an isolated Layer 2 (Ethernet) network on FABRIC and connect two compute nodes using **user-defined IP configuration**. Unlike auto-configuration, user-defined mode gives you full control over IP addresses, subnets, and routing -- essential for advanced network experiments.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create an isolated **Layer 2 (L2) Ethernet** network on a single FABRIC site
2. Add NIC components to nodes and retrieve their interfaces
3. Use **user-defined configuration** mode to manually assign IP addresses and subnets
4. Verify connectivity between nodes using `ping`
5. Clean up slices and release resources

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook to set up your `fabric_rc` and `ssh_config` files
2. Be familiar with creating basic slices (see [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb))

**Tip:** If you are new to FABRIC networking, start with the auto-configured L2 network notebook first, then return here to learn about user-defined configuration.

</div>

## Background: L2 Networks and User-Defined Configuration

### What is a Layer 2 Network?

A Layer 2 (L2) network on FABRIC is an isolated Ethernet broadcast domain. Nodes connected to the same L2 network can communicate using Ethernet frames, just as if they were plugged into the same physical switch. FABRIC creates these isolated networks using VLANs on its switching infrastructure.

### Local vs. Wide-Area L2 Networks

When all nodes are on the **same site**, FABRIC creates a **local** L2 network -- traffic stays within that site's switch. For nodes on **different sites**, see the [Wide-Area L2 Network](../create_l2network_wide_area/create_l2network_wide_area_config.ipynb) notebook.

### User-Defined vs. Auto Configuration

FABRIC supports two interface configuration modes:

- **Auto (`auto`)**: FABlib automatically assigns IP addresses from the subnet during post-boot configuration
- **User-defined (`config`)**: You specify exact IP addresses before submitting the slice. FABlib applies your configuration during post-boot setup

User-defined configuration is useful when you need predictable IP addresses (e.g., for firewall rules, DNS entries, or reproducible experiments).


### NIC Component Models

| Model | Description | Ports |
|-------|-------------|-------|
| `NIC_Basic` | 100 Gbps Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device | 2 |
| `NIC_ConnectX_6` | 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device | 2 |

## What We're Building

In this notebook we will create two nodes on the same site connected by a local L2 Ethernet bridge.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

We start by importing the FABlib library along with Python's `ipaddress` module, which we need for defining subnets and IP addresses.

In [ ]:
# Import Python's ipaddress module for subnet and IP address manipulation
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

# Import the FABlib library and create a manager instance
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We choose a random FABRIC site and define names for our nodes and network. Since this is a **local** L2 network, both nodes will be placed on the same site.

In [ ]:
# Name for the slice -- must be unique among your active slices
slice_name = 'MySlice'

# Pick a random FABRIC site with available resources
site = fablib.get_random_site()
print(f"Site: {site}")

# Node and network names
node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'

## Step 3: Create the Slice with User-Defined Network Configuration

This is the core step. We will:

1. Create a new slice
2. Add an L2 network with a specific subnet (`192.168.1.0/24`)
3. Add two nodes, each with a `NIC_Basic` component
4. Set each interface to **`config`** mode (user-defined) and assign a specific IP address
5. Submit the slice

<div class="fab-danger">

**Important:** The `set_mode('config')` call tells FABlib to use your manually specified IP address rather than auto-assigning one. You **must** call `set_ip_addr()` on each interface when using `config` mode, otherwise the interface will not be configured.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# Add an L2 network with a user-defined subnet
# The subnet parameter defines the IP range for this network
net1 = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

# --- Node 1 ---
node1 = slice.add_node(name=node1_name, site=site)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set mode to 'config' to enable user-defined IP assignment
iface1.set_mode('config')
# Connect the interface to our L2 network
net1.add_interface(iface1)
# Assign a specific IP address to this interface
iface1.set_ip_addr(IPv4Address("192.168.1.1"))

# --- Node 2 ---
node2 = slice.add_node(name=node2_name, site=site)
# Add a NIC_Basic component and get its first (only) interface
iface2 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set mode to 'config' to enable user-defined IP assignment
iface2.set_mode('config')
# Connect the interface to our L2 network
net1.add_interface(iface2)
# Assign a specific IP address to this interface
iface2.set_ip_addr(IPv4Address("192.168.1.2"))

# Submit the slice request to FABRIC
# This blocks until the slice is ready (~2-5 minutes)
slice.submit();

<div class="fab-success">

**What just happened?** FABRIC provisioned two VMs on the same site, created an isolated Ethernet segment between them, and configured each interface with the IP address you specified. Because you used `config` mode, FABlib applied your exact IP assignments during post-boot configuration.

</div>

## Step 4: Run the Experiment

With user-defined configuration, the slice is ready for experimentation immediately after it becomes active. We will verify connectivity by pinging Node2 from Node1.

<div class="fab-warning">

**Tip:** User-defined configuration works especially well when saving slices to a file and re-instantiating them later. Configuration tasks are stored in the saved slice, reducing the complexity of your notebooks and runtime steps.

</div>

In [ ]:
# Retrieve the slice (useful if reconnecting in a new session)
slice = fablib.get_slice(slice_name)

# Get references to both nodes
node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

# Look up Node2's IP address on the network (should be 192.168.1.2)
node2_addr = node2.get_interface(network_name=network_name).get_ip_addr()

# Ping Node2 from Node1 to verify L2 connectivity
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `ping` fails between nodes | Interface not configured | Verify both interfaces have `set_mode('config')` and `set_ip_addr()` called before `submit()` |
| `ping` shows 100% packet loss | Wrong subnet or IP mismatch | Ensure both IPs are in the same `/24` subnet (e.g., `192.168.1.x`) |
| Slice stuck in `Configuring` | Site may be busy or down | Try a different site by changing the `site` variable |
| `No resources available` error | Site lacks NIC_Basic capacity | Use `fablib.list_sites()` to find a site with available SmartNIC VFs |
| Interface has no IP after boot | Forgot `set_mode('config')` | Mode defaults to `auto`; explicitly set `config` mode for manual IPs |
| `submit()` times out | Network or demand issue | Retry with `slice.submit(wait_timeout=600)` for a longer timeout |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_site()` | Select a random FABRIC site | [get_random_site](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_site) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_l2network(name, subnet)` | Add a Layer 2 network to the slice | [add_l2network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l2network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `node.add_component(model, name)` | Add a NIC or other component to a node | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `component.get_interfaces()` | Get the list of interfaces on a component | [get_interfaces](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.get_interfaces) |
| `iface.set_mode(mode)` | Set interface configuration mode (`auto` or `config`) | [set_mode](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_mode) |
| `iface.set_ip_addr(addr)` | Assign an IP address to the interface | [set_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_ip_addr) |
| `net.add_interface(iface)` | Connect an interface to a network | [add_interface](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.add_interface) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you can create L2 networks with user-defined configuration, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Wide-Area L2 Network** | [create_l2network_wide_area_config](../create_l2network_wide_area/create_l2network_wide_area_config.ipynb) | Connect nodes across different FABRIC sites with L2 networking |
| **L2 with Explicit Routes** | [create_l2network_wide_area_ero_auto](../create_l2network_wide_area/create_l2network_wide_area_ero_auto.ipynb) | Control network paths with Explicit Route Options (ERO) |
| **FABnet IPv4 (L3)** | [create_l3network_fabnet_ipv4_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Use FABRIC's managed Layer 3 networking |
| **DHCP on L2 Networks** | [dhcp_l2_network](../dhcp_l2_network/dhcp.ipynb) | Set up a DHCP server on an L2 network |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |